<a href="https://colab.research.google.com/github/marianoInsa/dimiasa-models/blob/main/notebooks/05-E03-Preparacion-Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0 · Instalación y librerías

In [ ]:
%pip install numpy pandas scipy pyarrow azure-storage-file-datalake --quiet

In [ ]:
import io
import json
import warnings
import math
from datetime import datetime, timezone
from math import gcd

import numpy as np
import pandas as pd
from scipy.signal import resample_poly
from google.colab import userdata
from azure.storage.filedatalake import DataLakeServiceClient

warnings.filterwarnings("ignore")

# ── Constantes globales ───────────────────────────────────────────────────────
FS_TARGET      = 100    # Hz  ← 50 → 100 Hz
KAISER_BETA    = 5.0    # mismo que E02
WINDOW_SAMPLES = 200    # 2 s × 100 Hz  ← actualizado
OVERLAP_STEP   = 100    # 50 % de overlap  ← actualizado
FALL_LABEL_THR = 0.30   # umbral para etiquetar ventana como Fall

# Umbrales de descarte (lógica OR)
THR_PEARSON_MIN    = 0.85
THR_PHASE_MS_MAX   = 100.0
THR_ATTEN_PCT_MAX  = 25.0

DATASETS_META = {
    "KFall":    {"fs": 100, "csv": "KFall-Reduced.csv"},
    "UPFall":   {"fs": 100, "csv": "UPFall-Reduced.csv"},
    "SisFall":  {"fs": 200, "csv": "SisFall-Reduced.csv"},
    "FallAllD": {"fs": 238, "csv": "FallAllD-Reduced.csv"},
}

SPLIT_RATIOS = {"train": 0.70, "val": 0.10, "test": 0.20}

print("Configuración:")
print(f"  FS_TARGET          = {FS_TARGET} Hz")
print(f"  KAISER_BETA        = {KAISER_BETA}")
print(f"  WINDOW_SAMPLES     = {WINDOW_SAMPLES}  ({WINDOW_SAMPLES/FS_TARGET:.1f} s)")
print(f"  OVERLAP_STEP       = {OVERLAP_STEP}  ({OVERLAP_STEP/FS_TARGET:.1f} s)")
print(f"  FALL_LABEL_THR     = {FALL_LABEL_THR*100:.0f} %")
print(f"  Umbral Pearson r   ≥ {THR_PEARSON_MIN}")
print(f"  Umbral phase shift ≤ {THR_PHASE_MS_MAX} ms")
print(f"  Umbral atenuación  ≤ {THR_ATTEN_PCT_MAX} %")
print()
print("Factores de resampleo por dataset:")
for ds, meta in DATASETS_META.items():
    fs = meta["fs"]
    g  = gcd(FS_TARGET, fs)
    up, down = FS_TARGET // g, fs // g
    if up == down == 1:
        print(f"  {ds:12s}  {fs} Hz → {FS_TARGET} Hz  (sin resampleo — misma frecuencia)")
    else:
        print(f"  {ds:12s}  {fs} Hz → {FS_TARGET} Hz  (up={up}, down={down})")

## 1 · Validación de integridad de etiquetas

Verifica que `Activity_Label` contenga únicamente `"Fall"` y `"ADL"`, que no haya trials
con mezcla incoherente de etiquetas, y obtiene la distribución base antes del balanceo.

In [ ]:
# ── Conexión a Azure ──────────────────────────────────────────────────────────
CONNECTION_STRING = userdata.get("cadenaAzure")
service_client    = DataLakeServiceClient.from_connection_string(CONNECTION_STRING)
fs_bronce         = service_client.get_file_system_client("bronce")
dir_falls_bronze  = fs_bronce.get_directory_client("falls")

def _load_csv_bronze(filename: str) -> pd.DataFrame | None:
    try:
        content = dir_falls_bronze.get_file_client(filename).download_file().readall()
        df = pd.read_csv(io.BytesIO(content))
        print(f"  ✓  {filename:35s}  →  {len(df):>8,} filas")
        return df
    except Exception as e:
        print(f"  ✗  {filename:35s}  →  ERROR: {e}")
        return None

print("Descargando datasets desde bronce…")
raw_datasets: dict[str, pd.DataFrame] = {}
for ds_name, meta in DATASETS_META.items():
    df = _load_csv_bronze(meta["csv"])
    if df is not None:
        raw_datasets[ds_name] = df

print(f"\nDatasets cargados: {list(raw_datasets.keys())}")

In [ ]:
VALID_LABELS = {"Fall", "ADL"}

label_issues    = {}   # datasets con valores inesperados
mixed_issues    = {}   # trials con mezcla incoherente
label_dist_rows = []   # distribución base

for ds_name, df in raw_datasets.items():
    unique_labels = set(df["Activity_Label"].unique())
    unexpected    = unique_labels - VALID_LABELS

    if unexpected:
        label_issues[ds_name] = unexpected

    # Detectar trials donde el mismo (Subject, Activity_Code, Trial)
    # tiene filas de ambas etiquetas — anomalía potencial en Fall trials
    trials = df.groupby(["Subject", "Activity_Code", "Trial"])["Activity_Label"].nunique()
    mixed  = trials[trials > 1]
    if not mixed.empty:
        mixed_issues[ds_name] = mixed

    # Distribución base: trials únicos por label
    trial_ids = df.groupby(
        ["Subject", "Activity_Code", "Trial"]
    )["Activity_Label"].first().reset_index()

    for label in ("Fall", "ADL"):
        n = (trial_ids["Activity_Label"] == label).sum()
        label_dist_rows.append({
            "Dataset": ds_name,
            "Label":   label,
            "Trials":  n
        })

# ── Informe ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("  RESULTADO: Integridad de etiquetas")
print("=" * 60)

if label_issues:
    print("\n  ⚠️  Valores inesperados en Activity_Label:")
    for ds, vals in label_issues.items():
        print(f"     {ds}: {vals}")
else:
    print("\n  ✅  Todos los datasets contienen únicamente 'Fall' y 'ADL'.")

if mixed_issues:
    print("\n  ⚠️  Trials con mezcla de etiquetas Fall+ADL:")
    for ds, mixed in mixed_issues.items():
        print(f"     {ds}: {len(mixed)} trial(s) afectados")
        print(mixed.head(5).to_string())
else:
    print("  ✅  Sin trials con mezcla incoherente de etiquetas.")

print("\n  Distribución base de trials por dataset:")
df_dist = pd.DataFrame(label_dist_rows).pivot(
    index="Dataset", columns="Label", values="Trials"
).fillna(0).astype(int)
df_dist["Total"] = df_dist.sum(axis=1)
df_dist["Ratio ADL/Fall"] = (df_dist.get("ADL", 0) / df_dist.get("Fall", 1)).round(2)
print(df_dist.to_string())

## 2 · Filtrado de trials por calidad de resampleo

Carga el DataFrame de métricas desde Azure (`plata/falls/resampling_validation_results_100hz.csv`)
y lo usa para identificar los trials que superan los umbrales de descarte.
Genera `trial_quality_config.json` con los IDs de trials válidos por dataset.

In [ ]:
# ── Carga del CSV de métricas desde Azure (plata/falls/) ─────────────────────
#
# El archivo contiene las métricas calculadas en E02 para todos los datasets
# a 100 Hz. Columnas esperadas:
#   dataset, sensor, pearson_r, phase_shift_ms, peak_atten_pct
#   (y opcionalmente: snr_db, dtw_norm)
#
# El CSV tiene una fila por (dataset, trial, sensor). Los trials están en el
# mismo orden en que run_global_validation los procesó en E02 (order de
# drop_duplicates sobre Subject, Activity_Code, Trial dentro de los Fall).

fs_plata    = service_client.get_file_system_client("plata")
dir_metrics = fs_plata.get_directory_client("falls")

METRICS_FILENAME = "resampling_validation_results_100hz.csv"

try:
    content    = dir_metrics.get_file_client(METRICS_FILENAME).download_file().readall()
    df_metrics_raw = pd.read_csv(io.BytesIO(content))
    print(f"✅  {METRICS_FILENAME} cargado  →  {len(df_metrics_raw):,} filas")
    print(f"    Columnas: {list(df_metrics_raw.columns)}")
    print(f"    Datasets presentes: {sorted(df_metrics_raw['dataset'].unique())}")
    print(f"    Sensores presentes: {sorted(df_metrics_raw['sensor'].unique())}")
except Exception as e:
    raise RuntimeError(
        f"No se pudo cargar {METRICS_FILENAME} desde plata/falls/.\n"
        f"Error: {e}\n"
        "Verificá que el archivo exista y que la cadena de conexión sea correcta."
    )

# Separar métricas ACC por dataset para usarlas en filter_valid_trials
METRICS_BY_DS: dict[str, pd.DataFrame] = {}
for ds_name in DATASETS_META:
    mask = (
        (df_metrics_raw["dataset"] == ds_name) &
        (df_metrics_raw["sensor"]  == "acc")
    )
    sub = df_metrics_raw[mask].reset_index(drop=True)
    if sub.empty:
        print(f"  ⚠️  Sin métricas ACC para {ds_name} en el CSV")
    else:
        METRICS_BY_DS[ds_name] = sub
        print(f"  {ds_name:12s}  →  {len(sub):,} filas ACC")

In [ ]:
# ── Función: generar lista de IDs válidos para un dataset ────────────────────
def filter_valid_trials(
    df_raw: pd.DataFrame,
    df_metrics_acc: pd.DataFrame,
    pearson_min: float = THR_PEARSON_MIN,
    phase_ms_max: float = THR_PHASE_MS_MAX,
    atten_pct_max: float = THR_ATTEN_PCT_MAX,
) -> tuple[list, int, int]:
    """
    Cruza los IDs de trials del CSV raw con las métricas de ACC cargadas
    desde plata/falls/resampling_validation_results_100hz.csv.

    Las métricas están en el mismo orden en que run_global_validation de E02
    iteró los trials Fall (drop_duplicates sobre Subject, Activity_Code, Trial).
    La alineación se hace por posición — el orden debe coincidir.

    Retorna (valid_ids, n_valid, n_discarded) donde valid_ids es una lista
    de [Subject, Activity_Code, Trial] de los trials que pasan los umbrales.
    """
    # Obtener los IDs de trials en el mismo orden que E02 los procesó
    falls = df_raw[df_raw["Activity_Label"] == "Fall"]
    trial_ids = (
        falls[["Subject", "Activity_Code", "Trial"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    acc_metrics = df_metrics_acc.reset_index(drop=True)

    n_trials      = len(trial_ids)
    n_metrics_acc = len(acc_metrics)

    if n_trials != n_metrics_acc:
        print(
            f"  ⚠️  Discrepancia de longitud: {n_trials} trials en CSV vs "
            f"{n_metrics_acc} filas de métricas ACC. "
            f"Se usará min({n_trials}, {n_metrics_acc})."
        )
        n_use       = min(n_trials, n_metrics_acc)
        trial_ids   = trial_ids.iloc[:n_use].reset_index(drop=True)
        acc_metrics = acc_metrics.iloc[:n_use].reset_index(drop=True)

    # Aplicar criterios de descarte (lógica OR)
    discard_mask = (
        (acc_metrics["pearson_r"]      < pearson_min)  |
        (acc_metrics["phase_shift_ms"] > phase_ms_max) |
        (acc_metrics["peak_atten_pct"] > atten_pct_max)
    )

    valid_ids = (
        trial_ids[~discard_mask]
        .values.tolist()
    )

    n_valid     = len(valid_ids)
    n_discarded = int(discard_mask.sum())

    return valid_ids, n_valid, n_discarded

In [ ]:
# ── Generar config de calidad por dataset ────────────────────────────────────
quality_config = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "fs_target":    FS_TARGET,
    "criteria": {
        "pearson_r_min":      THR_PEARSON_MIN,
        "phase_shift_ms_max": THR_PHASE_MS_MAX,
        "peak_atten_pct_max": THR_ATTEN_PCT_MAX,
    },
    "datasets": {}
}

print("Filtrando trials por calidad de resampleo…\n")
print(f"  {'Dataset':12s}  {'Total':>7}  {'Válidos':>8}  {'Descartados':>12}  {'% Descarte':>10}")
print("  " + "-" * 55)

for ds_name, meta in DATASETS_META.items():
    if ds_name not in raw_datasets:
        print(f"  {ds_name:12s}  — OMITIDO (sin datos raw)")
        continue
    if ds_name not in METRICS_BY_DS:
        print(f"  {ds_name:12s}  — OMITIDO (sin métricas)")
        continue

    df_raw  = raw_datasets[ds_name]
    df_acc  = METRICS_BY_DS[ds_name]

    # Total de trials Fall en el raw
    falls       = df_raw[df_raw["Activity_Label"] == "Fall"]
    total_falls = falls[["Subject", "Activity_Code", "Trial"]].drop_duplicates().shape[0]

    valid_ids, n_valid, n_discarded = filter_valid_trials(df_raw, df_acc)

    pct_discard = n_discarded / total_falls * 100 if total_falls > 0 else 0

    quality_config["datasets"][ds_name] = {
        "total":     total_falls,
        "valid":     n_valid,
        "discarded": n_discarded,
        "valid_ids": valid_ids,
    }

    print(f"  {ds_name:12s}  {total_falls:>7,}  {n_valid:>8,}  {n_discarded:>12,}  {pct_discard:>9.1f}%")

print("\n  Filtrado completado.")

In [ ]:
# ── Desglose de motivos de descarte por dataset ───────────────────────────────
print("\nDesglose de motivos de descarte (lógica OR — un trial puede activar varios):\n")
print(f"  {'Dataset':12s}  {'r<0.85':>8}  {'Δt>100ms':>10}  {'Aten>25%':>10}")
print("  " + "-" * 46)

for ds_name in DATASETS_META:
    if ds_name not in raw_datasets or ds_name not in METRICS_BY_DS:
        continue

    df_raw = raw_datasets[ds_name]
    df_acc = METRICS_BY_DS[ds_name].reset_index(drop=True)

    falls    = df_raw[df_raw["Activity_Label"] == "Fall"]
    n_trials = falls[["Subject", "Activity_Code", "Trial"]].drop_duplicates().shape[0]
    n_use    = min(n_trials, len(df_acc))
    df_acc   = df_acc.iloc[:n_use]

    n_pearson = int((df_acc["pearson_r"]      < THR_PEARSON_MIN).sum())
    n_phase   = int((df_acc["phase_shift_ms"] > THR_PHASE_MS_MAX).sum())
    n_atten   = int((df_acc["peak_atten_pct"] > THR_ATTEN_PCT_MAX).sum())

    print(f"  {ds_name:12s}  {n_pearson:>8,}  {n_phase:>10,}  {n_atten:>10,}")

In [ ]:
# ── Serializar y subir JSON a Azure ──────────────────────────────────────────
def _to_native(obj):
    """Convierte recursivamente tipos numpy a tipos Python nativos."""    if isinstance(obj, dict):
        return {k: _to_native(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_native(i) for i in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

quality_config_native = _to_native(quality_config)
json_bytes = json.dumps(quality_config_native, indent=2, ensure_ascii=False).encode("utf-8")

fs_gold = service_client.get_file_system_client("gold")
try:
    fs_gold.create_directory("falls")
except Exception:
    pass

gold_dir    = fs_gold.get_directory_client("falls")
file_client = gold_dir.get_file_client("trial_quality_config.json")
file_client.upload_data(json_bytes, overwrite=True, length=len(json_bytes))

print(f"✅  trial_quality_config.json subido a gold/falls/  ({len(json_bytes):,} bytes)")
print(f"    Generado: {quality_config['generated_at']}")
print(f"    fs_target: {quality_config['fs_target']} Hz")

for ds_name, info in quality_config_native["datasets"].items():
    print(f"    {ds_name:12s}  valid={info['valid']:>4}  discarded={info['discarded']:>4}")

## 3 · Resampleo definitivo y guardado en Parquet

Para cada dataset:
1. Cargar CSV desde `bronce/falls/`
2. Filtrar trials según `valid_ids` del JSON de Sección 2
3. Resamplear cada trial completo con `resample_poly` (Kaiser β=5.0)
   - KFall / UPFall (100 Hz): sin resampleo — ya están a 100 Hz
   - SisFall (200 Hz): down=2
   - FallAllD (238 Hz): up=50, down=119
4. Guardar en `gold/falls/dataset=<nombre>/part-0.parquet`

In [ ]:
def get_poly_factors(fs_orig: int, fs_target: int) -> tuple[int, int]:
    g = gcd(fs_target, fs_orig)
    return fs_target // g, fs_orig // g


def resample_trial_df(trial_df: pd.DataFrame, fs_orig: int) -> pd.DataFrame:
    """
    Resamplea un trial completo columna a columna y reconstruye el DataFrame.
    Preserva Subject, Activity_Code, Activity_Label, Trial intactos.
    Recalcula Sample_Index como secuencia 0…N-1 tras el resampleo.

    Si fs_orig == FS_TARGET no aplica resampleo — devuelve el trial con
    Sample_Index reindexado.
    """
    meta_cols = ["Subject", "Activity_Code", "Activity_Label", "Trial"]
    sig_cols  = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]

    meta = {c: trial_df[c].iloc[0] for c in meta_cols}

    if fs_orig == FS_TARGET:
        # Sin resampleo: copiar señales directamente
        out = trial_df[sig_cols].copy().reset_index(drop=True)
    else:
        up, down = get_poly_factors(fs_orig, FS_TARGET)
        resampled = {}
        for col in sig_cols:
            signal = trial_df[col].values.astype(float)
            resampled[col] = resample_poly(signal, up=up, down=down,
                                           window=("kaiser", KAISER_BETA))
        out = pd.DataFrame(resampled)

    for c in meta_cols:
        out[c] = meta[c]
    out["Sample_Index"] = np.arange(len(out))

    col_order = meta_cols + ["Sample_Index"] + sig_cols
    return out[col_order]


print("Funciones de resampleo cargadas.")
print()
print("Verificación de factores por dataset:")
for ds, meta in DATASETS_META.items():
    fs = meta["fs"]
    if fs == FS_TARGET:
        print(f"  {ds:12s}  {fs} Hz = {FS_TARGET} Hz  → sin resampleo")
    else:
        up, down = get_poly_factors(fs, FS_TARGET)
        print(f"  {ds:12s}  {fs} Hz → {FS_TARGET} Hz  (up={up}, down={down})")

In [ ]:
# ── Pipeline de resampleo + guardado ─────────────────────────────────────────
SCHEMA_COLS = [
    "Subject", "Activity_Code", "Activity_Label", "Trial", "Sample_Index",
    "Ax", "Ay", "Az", "Gx", "Gy", "Gz"
]

resample_summary = {}

for ds_name, meta in DATASETS_META.items():
    if ds_name not in raw_datasets:
        print(f"[{ds_name}] OMITIDO — no disponible en memoria")
        continue
    if ds_name not in quality_config_native["datasets"]:
        print(f"[{ds_name}] OMITIDO — sin entrada en trial_quality_config")
        continue

    fs_orig   = meta["fs"]
    df_raw    = raw_datasets[ds_name]
    valid_ids = quality_config_native["datasets"][ds_name]["valid_ids"]
    up, down  = get_poly_factors(fs_orig, FS_TARGET)

    if fs_orig == FS_TARGET:
        print(f"\n[{ds_name}]  {fs_orig} Hz = {FS_TARGET} Hz  (sin resampleo)")
    else:
        print(f"\n[{ds_name}]  {fs_orig} Hz → {FS_TARGET} Hz  (up={up}, down={down})")
    print(f"  Trials válidos (Fall): {len(valid_ids):,}")

    valid_set = {(row[0], row[1], row[2]) for row in valid_ids}

    df_fall = df_raw[df_raw["Activity_Label"] == "Fall"]
    df_adl  = df_raw[df_raw["Activity_Label"] == "ADL"]

    df_fall_filtered = df_fall[
        df_fall.apply(
            lambda r: (r["Subject"], r["Activity_Code"], r["Trial"]) in valid_set,
            axis=1
        )
    ]

    df_to_resample = pd.concat([df_fall_filtered, df_adl], ignore_index=True)

    trial_groups     = df_to_resample.groupby(
        ["Subject", "Activity_Code", "Trial"], sort=False
    )
    resampled_chunks = []
    n_trials_total   = len(trial_groups)
    n_done           = 0

    for (subj, act, trial), grp in trial_groups:
        try:
            resampled_chunks.append(resample_trial_df(grp.copy(), fs_orig))
        except Exception as exc:
            print(f"  ⚠️  Error en trial ({subj}, {act}, {trial}): {exc}")
        n_done += 1
        if n_done % 200 == 0 or n_done == n_trials_total:
            print(f"  Progreso: {n_done}/{n_trials_total} trials", end="\r")

    print()

    df_out = pd.concat(resampled_chunks, ignore_index=True)

    missing = [c for c in SCHEMA_COLS if c not in df_out.columns]
    if missing:
        raise ValueError(f"[{ds_name}] Columnas faltantes en la salida: {missing}")

    df_out = df_out[SCHEMA_COLS]

    n_fall_out = (df_out["Activity_Label"] == "Fall").sum()
    n_adl_out  = (df_out["Activity_Label"] == "ADL").sum()
    print(f"  Filas Fall: {n_fall_out:,}  |  ADL: {n_adl_out:,}  |  Total: {len(df_out):,}")

    parquet_buffer = io.BytesIO()
    df_out.to_parquet(parquet_buffer, index=False, engine="pyarrow")
    parquet_bytes  = parquet_buffer.getvalue()

    dataset_dir = f"falls/dataset={ds_name}"
    try:
        fs_gold.create_directory(dataset_dir)
    except Exception:
        pass

    dir_client  = fs_gold.get_directory_client(dataset_dir)
    file_client = dir_client.get_file_client("part-0.parquet")
    file_client.upload_data(parquet_bytes, overwrite=True, length=len(parquet_bytes))

    size_mb = len(parquet_bytes) / 1024 / 1024
    print(f"  ✅  Subido → gold/{dataset_dir}/part-0.parquet  ({size_mb:.1f} MB)")

    resample_summary[ds_name] = {
        "rows_fall": int(n_fall_out),
        "rows_adl":  int(n_adl_out),
        "size_mb":   round(size_mb, 2),
    }

print("\n" + "="*55)
print("  Resampleo y carga a Parquet completados.")

## 4 · Split por sujeto y análisis de balance

**Orden de operaciones — no negociable:**
1. Determinar asignación de sujetos a Train/Val/Test por dataset
2. Guardar `subject_split_config.json`
3. Generar ventanas solo para los sujetos de cada split
4. Analizar balance de clases y aplicar undersampling ADL

In [ ]:
# ── Carga de los Parquets desde gold para trabajar con datos resampleados ────
print("Cargando Parquets desde gold/falls/…")
gold_datasets: dict[str, pd.DataFrame] = {}

for ds_name in DATASETS_META:
    try:
        dir_client = fs_gold.get_directory_client(f"falls/dataset={ds_name}")
        content    = dir_client.get_file_client("part-0.parquet").download_file().readall()
        df         = pd.read_parquet(io.BytesIO(content))
        gold_datasets[ds_name] = df
        print(f"  ✓  {ds_name:12s}  {len(df):>9,} filas")
    except Exception as e:
        print(f"  ✗  {ds_name:12s}  ERROR: {e}")

print("Carga completada.")

In [ ]:
# ── Función de split estratificado por sujeto ────────────────────────────────
import random

def split_subjects_stratified(
    subjects: list,
    ratios: dict,
    seed: int = 42,
) -> dict[str, list]:
    """
    Asigna sujetos a Train/Val/Test de forma estratificada (aleatoria con seed).
    Garantiza que todos los sujetos queden asignados.
    """
    rng = random.Random(seed)
    shuffled = sorted(subjects)
    rng.shuffle(shuffled)

    n = len(shuffled)
    n_train = round(n * ratios["train"])
    n_val   = round(n * ratios["val"])
    n_test  = n - n_train - n_val

    return {
        "train": shuffled[:n_train],
        "val":   shuffled[n_train : n_train + n_val],
        "test":  shuffled[n_train + n_val :],
    }


# ── Generar asignación de sujetos ────────────────────────────────────────────
split_config = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "split_ratios": SPLIT_RATIOS,
    "seed":         42,
    "datasets":     {}
}

print(f"  {'Dataset':12s}  {'Sujetos':>8}  {'Train':>7}  {'Val':>5}  {'Test':>6}")
print("  " + "-" * 46)

for ds_name, df in gold_datasets.items():
    subjects = sorted(df["Subject"].unique().tolist())
    split    = split_subjects_stratified(subjects, SPLIT_RATIOS)

    split_config["datasets"][ds_name] = {
        "total_subjects": len(subjects),
        "train": split["train"],
        "val":   split["val"],
        "test":  split["test"],
    }

    print(
        f"  {ds_name:12s}  {len(subjects):>8,}  "
        f"{len(split['train']):>7,}  {len(split['val']):>5,}  {len(split['test']):>6,}"
    )

split_json_bytes = json.dumps(
    _to_native(split_config), indent=2, ensure_ascii=False
).encode("utf-8")

dir_gold    = fs_gold.get_directory_client("falls")
file_client = dir_gold.get_file_client("subject_split_config.json")
file_client.upload_data(split_json_bytes, overwrite=True, length=len(split_json_bytes))

print(f"\n✅  subject_split_config.json subido a gold/falls/  ({len(split_json_bytes):,} bytes)")

In [ ]:
# ── Función de ventaneo ───────────────────────────────────────────────────────
SIGNAL_COLS = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]

def generate_windows(
    df: pd.DataFrame,
    window_samples: int = WINDOW_SAMPLES,
    step: int = OVERLAP_STEP,
    fall_threshold: float = FALL_LABEL_THR,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Genera ventanas deslizantes sobre los trials del DataFrame.
    Itera por trial (no concatena todo) para no crear ventanas que crucen
    el límite entre trials.

    Parámetros:
        window_samples : 200 muestras = 2 s a 100 Hz
        step           : 100 muestras = 50% de overlap
        fall_threshold : fracción mínima de muestras Fall para etiquetar
                         la ventana como Fall (default 0.30)

    Retorna:
        X : np.ndarray  shape (N, 200, 6)
        y : np.ndarray  shape (N,)  dtype int8  {0: ADL, 1: Fall}
    """
    X_list, y_list = [], []

    trial_groups = df.groupby(
        ["Subject", "Activity_Code", "Trial"], sort=False
    )

    for _, grp in trial_groups:
        sigs       = grp[SIGNAL_COLS].values.astype(np.float32)
        labels_bin = (grp["Activity_Label"].values == "Fall").astype(np.float32)

        n = len(sigs)
        start = 0
        while start + window_samples <= n:
            end = start + window_samples
            window_sig    = sigs[start:end]
            window_labels = labels_bin[start:end]

            fall_ratio = window_labels.mean()
            label = 1 if fall_ratio >= fall_threshold else 0

            X_list.append(window_sig)
            y_list.append(label)
            start += step

    X = np.stack(X_list) if X_list else np.empty((0, window_samples, len(SIGNAL_COLS)))
    y = np.array(y_list, dtype=np.int8)
    return X, y


print(f"Función de ventaneo cargada.")
print(f"  Ventana: {WINDOW_SAMPLES} muestras = {WINDOW_SAMPLES/FS_TARGET:.1f} s a {FS_TARGET} Hz")
print(f"  Overlap: {OVERLAP_STEP} muestras = {OVERLAP_STEP/FS_TARGET:.1f} s")

In [ ]:
# ── Generar ventanas por split y por dataset ─────────────────────────────────
split_windows: dict[str, dict[str, tuple]] = {}
window_stats_rows = []

for ds_name, df in gold_datasets.items():
    split_windows[ds_name] = {}
    subjects_split = split_config["datasets"][ds_name]

    print(f"\n[{ds_name}]")

    for split_name in ("train", "val", "test"):
        split_subjects = subjects_split[split_name]
        df_split = df[df["Subject"].isin(split_subjects)]

        X, y = generate_windows(df_split)

        n_fall = int((y == 1).sum())
        n_adl  = int((y == 0).sum())
        ratio  = n_adl / n_fall if n_fall > 0 else float("inf")

        split_windows[ds_name][split_name] = (X, y)

        print(
            f"  {split_name:6s}  ventanas={len(y):>7,}  "
            f"Fall={n_fall:>6,}  ADL={n_adl:>7,}  "
            f"ratio ADL/Fall={ratio:>5.1f}"
        )

        window_stats_rows.append({
            "Dataset": ds_name,
            "Split":   split_name,
            "Total":   len(y),
            "Fall":    n_fall,
            "ADL":     n_adl,
            "Ratio_ADL_Fall": round(ratio, 2),
        })

df_window_stats = pd.DataFrame(window_stats_rows)

In [ ]:
# ── Análisis de balance global ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.linestyle": "--", "grid.alpha": 0.45})

print("\n" + "=" * 75)
print("  BALANCE DE VENTANAS POR SPLIT Y DATASET")
print("=" * 75)
print(df_window_stats.to_string(index=False))

global_agg = df_window_stats.groupby("Split")[["Total", "Fall", "ADL"]].sum()
global_agg["Ratio_ADL_Fall"] = (global_agg["ADL"] / global_agg["Fall"]).round(2)
print("\n  Agregado global (todos los datasets):")
print(global_agg.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Balance de clases por split — antes de balanceo", fontsize=13)

splits   = ["train", "val", "test"]
ds_names = list(gold_datasets.keys())
x        = np.arange(len(ds_names))
width    = 0.35

for ax, split_name in zip(axes, splits):
    sub  = df_window_stats[df_window_stats["Split"] == split_name]
    fall = [sub[sub["Dataset"] == ds]["Fall"].values[0] if ds in sub["Dataset"].values else 0 for ds in ds_names]
    adl  = [sub[sub["Dataset"] == ds]["ADL"].values[0]  if ds in sub["Dataset"].values else 0 for ds in ds_names]

    ax.bar(x - width/2, fall, width, label="Fall",  color="tomato",    alpha=0.85)
    ax.bar(x + width/2, adl,  width, label="ADL",   color="steelblue", alpha=0.85)
    ax.set_title(split_name.capitalize(), fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(ds_names, rotation=15, fontsize=9)
    ax.legend(fontsize=9)
    ax.set_ylabel("Nº ventanas")

plt.tight_layout()
plt.show()

In [ ]:
# ── Decisión de estrategia de balanceo (undersampling ADL) ───────────────────
TARGET_RATIO = 3.0   # objetivo ADL/Fall ≈ 1:3

print(f"Objetivo de ratio ADL/Fall ≤ {TARGET_RATIO:.1f}\n")
print(f"  {'Dataset':12s}  {'Split':6s}  {'Fall':>7}  {'ADL actual':>12}  {'ADL→target':>12}  {'Reducción':>10}")
print("  " + "-" * 58)

undersample_plan = {}

for ds_name in ds_names:
    undersample_plan[ds_name] = {}
    for split_name in splits:
        sub = df_window_stats[
            (df_window_stats["Dataset"] == ds_name) &
            (df_window_stats["Split"]   == split_name)
        ]
        if sub.empty:
            continue

        n_fall     = int(sub["Fall"].values[0])
        n_adl      = int(sub["ADL"].values[0])
        target_adl = int(n_fall * TARGET_RATIO)
        target_adl = min(target_adl, n_adl)
        reduction  = (n_adl - target_adl) / n_adl * 100 if n_adl > 0 else 0

        undersample_plan[ds_name][split_name] = {
            "n_fall":       n_fall,
            "n_adl_orig":   n_adl,
            "n_adl_target": target_adl,
        }

        print(
            f"  {ds_name:12s}  {split_name:6s}  {n_fall:>7,}  "
            f"{n_adl:>12,}  {target_adl:>12,}  {reduction:>9.1f}%"
        )

print("\nNota: undersampling aleatorio sobre ventanas ADL por trial (no por muestras).")
print("SMOTE temporal queda como opción de segunda iteración si el modelo no converge.")

In [ ]:
# ── Preview rápido del tensor de ventanas (Train combinado) ───────────────────
# Solo verifica shapes y dtypes. No carga todo en RAM si los datasets son grandes.

print("\nVerificación de shapes de los tensores generados:")
print(f"  {'Dataset':12s}  {'Split':6s}  {'X.shape':>22}  {'y.shape':>14}  {'y dtype':>8}")
print("  " + "-" * 66)

for ds_name, splits_dict in split_windows.items():
    for split_name, (X, y) in splits_dict.items():
        print(
            f"  {ds_name:12s}  {split_name:6s}  "
            f"{str(X.shape):>22}  {str(y.shape):>14}  {str(y.dtype):>8}"
        )

print(f"\n  ✅  Shapes correctos: (N, {WINDOW_SAMPLES}, 6) — {WINDOW_SAMPLES} muestras × 6 canales IMU @ {FS_TARGET} Hz")

## Resumen de E03

| Artefacto | Destino Azure |
|---|---|
| `trial_quality_config.json` | `gold/falls/` |
| `subject_split_config.json` | `gold/falls/` |
| `part-0.parquet` × 4 datasets | `gold/falls/dataset=<nombre>/` |

**Configuración aplicada:**
- Frecuencia objetivo: **100 Hz**
- Ventana: **200 muestras = 2 s**
- Overlap: **50% (paso de 100 muestras)**
- Umbral Fall: ≥ 30% de muestras Fall en la ventana
- Métricas de filtrado: cargadas desde `plata/falls/resampling_validation_results_100hz.csv`
- KFall y UPFall (100 Hz): sin resampleo aplicado
- SisFall (200 Hz): down=2
- FallAllD (238 Hz): up=50, down=119